In [ ]:
!pip install -q isodate streamlit
import json
import re
import isodate
import requests
from fractions import Fraction
from bs4 import BeautifulSoup
import streamlit as st

# =====================================================================
# 0. STREAMLIT SAYFA YAPILANDIRMASI
# =====================================================================
st.set_page_config(
    page_title="Bon Appétit | Akıllı Tarif & Kiler Asistanı",
    page_icon="🍳",
    layout="centered"
)

# =====================================================================
# 1. MATEMATİKSEL MALZEME ÖLÇEKLEME MOTORU
# =====================================================================
class MalzemeMotoru:
    @staticmethod
    def sayi_donustur(metin):
        try:
            if " " in metin:
                tam, kesir = metin.split(" ")
                return float(tam) + float(Fraction(kesir))
            if "/" in metin:
                return float(Fraction(metin))
            return float(metin)
        except Exception:
            return None

    @classmethod
    def olcekle(cls, malzeme_metni, carpan):
        if carpan == 1.0:
            return malzeme_metni

        desen = r'^(\d+\s+\d+/\d+|\d+/\d+|\d+(?:\.\d+)?)'
        eslesme = re.match(desen, malzeme_metni.strip())

        if eslesme:
            eski_sayi_str = eslesme.group(1)
            sayisal_deger = cls.sayi_donustur(eski_sayi_str)
            if sayisal_deger:
                yeni_deger = sayisal_deger * carpan
                formatli = f"{int(yeni_deger)}" if yeni_deger.is_integer() else f"{yeni_deger:.2f}".rstrip("0").rstrip(".")
                return re.sub(desen, formatli, malzeme_metni.strip(), count=1)
        return malzeme_metni


# =====================================================================
# 2. SCHEMA.ORG/RECIPE UYUMLU CLASS YAPISI
# =====================================================================
class Tarif:
    def __init__(self, name, ingredients, cook_time=None, prep_time=None,
                 total_time=None, recipe_yield=None, category=None,
                 cuisine=None, instructions=None, gorsel_url="", url=""):
        self.ad = name
        self.malzemeler = ingredients or []
        self.pisirme_suresi_raw = cook_time
        self.hazirlik_suresi_raw = prep_time
        self.toplam_sure_raw = total_time
        self.kisi_sayisi_str = self._parse_yield(recipe_yield)
        self.varsayilan_porsiyon = self._extract_servings_int(self.kisi_sayisi_str)
        self.kategori = category
        self.mutfak = cuisine
        self.talimatlar = instructions or []
        self.gorsel_url = gorsel_url
        self.url = url

        self.pisirme_suresi_dk = self._iso_to_minutes(self.pisirme_suresi_raw)
        self.hazirlik_suresi_dk = self._iso_to_minutes(self.hazirlik_suresi_raw)
        self.toplam_sure_dk = self._iso_to_minutes(self.toplam_sure_raw)

    @staticmethod
    def _extract_servings_int(yield_str):
        rakamlar = re.findall(r'\d+', str(yield_str))
        return int(rakamlar[0]) if rakamlar else 4

    @staticmethod
    def _iso_to_minutes(iso_duration):
        if not iso_duration:
            return None
        try:
            dur = isodate.parse_duration(iso_duration)
            return int(dur.total_seconds()) // 60
        except Exception:
            m = re.search(r'(\d+)\s*(?:dakika|min|m)', str(iso_duration), re.IGNORECASE)
            return int(m.group(1)) if m else None

    @staticmethod
    def _parse_yield(recipe_yield):
        if not recipe_yield:
            return "4 porsiyon"
        if isinstance(recipe_yield, list):
            return str(recipe_yield[0])
        return str(recipe_yield)

    @classmethod
    def from_schema_json(cls, data, backup_image="", url=""):
        raw_instructions = data.get("recipeInstructions", [])
        instructions = []
        if isinstance(raw_instructions, list):
            for step in raw_instructions:
                if isinstance(step, dict):
                    instructions.append(step.get("text", ""))
                elif isinstance(step, str):
                    instructions.append(step)
        elif isinstance(raw_instructions, str):
            instructions.append(raw_instructions)

        img_val = data.get("image")
        gorsel = ""
        if isinstance(img_val, str):
            gorsel = img_val
        elif isinstance(img_val, list) and len(img_val) > 0:
            first = img_val[0]
            gorsel = first if isinstance(first, str) else first.get("url", "")
        elif isinstance(img_val, dict):
            gorsel = img_val.get("url", "")

        if not gorsel:
            gorsel = backup_image

        return cls(
            name=data.get("name", "İsimsiz Tarif"),
            ingredients=data.get("recipeIngredient", []),
            cook_time=data.get("cookTime"),
            prep_time=data.get("prepTime"),
            total_time=data.get("totalTime"),
            recipe_yield=data.get("recipeYield"),
            category=data.get("recipeCategory"),
            cuisine=data.get("recipeCuisine"),
            instructions=instructions,
            gorsel_url=gorsel,
            url=url
        )

    def dinamik_malzeme_listesi(self, hedef_kisi):
        carpan = hedef_kisi / self.varsayilan_porsiyon
        return [MalzemeMotoru.olcekle(m, carpan) for m in self.malzemeler]

    def kiler_kontrolu(self, evdeki_malzemeler):
        if not evdeki_malzemeler:
            return 0, self.malzemeler, []

        evdekiler = [e.strip().lower() for e in evdeki_malzemeler if e.strip()]
        var_olanlar = []
        eksikler = []

        for m in self.malzemeler:
            m_lower = m.lower()
            if any(e in m_lower for e in evdekiler):
                var_olanlar.append(m)
            else:
                eksikler.append(m)

        oran = int((len(var_olanlar) / len(self.malzemeler)) * 100) if self.malzemeler else 0
        return oran, var_olanlar, eksikler

    def sure_bilgisi(self):
        if self.pisirme_suresi_dk:
            return f"{self.pisirme_suresi_dk} dakika (Pişirme)"
        elif self.toplam_sure_dk:
            return f"{self.toplam_sure_dk} dakika (Toplam)"
        elif self.hazirlik_suresi_dk:
            return f"{self.hazirlik_suresi_dk} dakika (Hazırlık)"
        return "Süre belirtilmemiş."


# =====================================================================
# 3. SCRAPING VE ÖNBELLEKLEME MOTORU
# =====================================================================
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, Scientific/124.0.0.0 Safari/537.36"
    )
}

tarif_linkleri = [
    "https://www.bonappetit.com/recipe/classic-french-toast",
    "https://www.bonappetit.com/recipe/creamy-white-bean-soup-with-kale-and-parmesan",
    "https://www.bonappetit.com/recipe/pasta-al-limone",
    "https://www.bonappetit.com/recipe/chocolate-chip-cookies",
    "https://www.bonappetit.com/recipe/focaccia-bread",
    "https://www.bonappetit.com/recipe/shakshuka",
    "https://www.bonappetit.com/recipe/cast-iron-roast-chicken-with-crispy-potatoes",
    "https://www.bonappetit.com/recipe/classic-guacamole",
    "https://www.bonappetit.com/recipe/brown-butter-toffee-chocolate-chip-cookies",
    "https://www.bonappetit.com/recipe/creamy-pasta-with-crispy-mushrooms"
]

@st.cache_resource
def tarifleri_yukle():
    tarif_havuzu = {}
    for link in tarif_linkleri:
        try:
            resp = requests.get(link, headers=HEADERS, timeout=10)
            if resp.status_code != 200:
                continue

            soup = BeautifulSoup(resp.content, "html.parser")
            backup_img = ""
            og_img_tag = soup.find("meta", property="og:image")
            if og_img_tag and og_img_tag.get("content"):
                backup_img = og_img_tag["content"]

            for s in soup.find_all("script", type="application/ld+json"):
                if not s.string:
                    continue
                try:
                    parsed = json.loads(s.string)
                    items = parsed if isinstance(parsed, list) else [parsed]
                    for item in items:
                        for node in item.get("@graph", [item]):
                            node_type = node.get("@type", "")
                            if node_type == "Recipe" or (isinstance(node_type, list) and "Recipe" in node_type):
                                t = Tarif.from_schema_json(node, backup_image=backup_img, url=link)
                                tarif_havuzu[t.ad] = t
                                break
                except Exception:
                    continue
        except Exception:
            continue
    return tarif_havuzu

tarifler_dict = tarifleri_yukle()

if not tarifler_dict:
    st.error("Tarifler yüklenirken bir bağlantı sorunu oluştu. Lütfen sayfayı yenileyin.")
    st.stop()


# =====================================================================
# 4. STREAMLIT ARAYÜZ (UI/UX) TASARIMI
# =====================================================================
st.markdown("""
<div style="background: linear-gradient(135deg, #0f172a, #1e293b); color: #f8fafc; padding: 20px; border-radius: 12px; margin-bottom: 20px; border-left: 6px solid #f97316;">
    <h2 style="margin: 0 0 6px 0; font-size: 22px; font-family: sans-serif;">🍳 Bon Appétit: Akıllı Porsiyon & Kiler Asistanı</h2>
    <p style="margin: 0; font-size: 13px; color: #94a3b8;">Kişi sayısına göre gramajları otomatik yeniden ölçekleyin, evdeki malzemelerle uyum oranını hesaplayın.</p>
</div>
""", unsafe_allow_html=True)

# Seçim Alanı
secilen_yemek = st.selectbox("📂 10 Niş Tariften Birini Seçin:", list(tarifler_dict.keys()))
secilen_tarif = tarifler_dict[secilen_yemek]

col1, col2 = st.columns(2)
with col1:
    hedef_kisi = st.slider("👥 Kişi Sayısı (Gramajları Ölçekle):", min_value=1, max_value=16, value=secilen_tarif.varsayilan_porsiyon)
with col2:
    kiler_input = st.text_input("🛒 Kiler/Dolap (Virgülle ayırın):", placeholder="Örn: egg, butter, garlic")

st.markdown("---")

# Sonuç Kartı Düzeni
col_img, col_info = st.columns([1, 1.4])

with col_img:
    if secilen_tarif.gorsel_url:
        st.image(secilen_tarif.gorsel_url, use_container_width=True)
    else:
        st.info("Görsel bulunamadı.")

with col_info:
    st.markdown(f"### {secilen_tarif.ad}")
    st.markdown(f"⏱️ **Süre:** {secilen_tarif.sure_bilgisi()}")
    st.markdown(f"👥 **Porsiyon:** {hedef_kisi} Kişilik *(Orijinal: {secilen_tarif.varsayilan_porsiyon} Kişi)*")

    # Kiler Kontrolü
    evdeki_girdiler = [x.strip() for x in kiler_input.split(",") if x.strip()]
    oran, bulunanlar, eksikler = secilen_tarif.kiler_kontrolu(evdeki_girdiler)

    if evdeki_girdiler:
        st.markdown(f"**Kiler Uyumluluğu:** %{oran} ({len(bulunanlar)} hazır / {len(eksikler)} eksik)")
        st.progress(oran / 100)

# Dinamik Ölçekli Malzeme Listesi
st.markdown(f"### 🛒 {hedef_kisi} Kişilik Porsiyona Göre Güncellenmiş Malzemeler:")
olcekli_malzemeler = secilen_tarif.dinamik_malzeme_listesi(hedef_kisi)

for m in olcekli_malzemeler:
    m_lower = m.lower()
    evde_var = any(e.lower() in m_lower for e in evdeki_girdiler) if evdeki_girdiler else False
    if evde_var:
        st.markdown(f"✅ <span style='color: #15803d; font-weight: 600;'>{m}</span> <small style='color: #166534; background: #dcfce7; padding: 1px 6px; border-radius: 4px;'>Dolapta Var</small>", unsafe_allow_html=True)
    else:
        st.markdown(f"• {m}")

2026-10-12 12:29:34.035 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-12 12:29:34.044 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-12 12:29:34.045 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-12 12:29:34.046 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-12 12:29:34.047 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-12 12:29:34.048 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-12 12:29:34.049 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-12 12:29:34.050 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar